In [1]:
import os
import sys
import logging
from datetime import datetime  # 1. 导入 datetime 模块
import pandas as pd
from make_ids_map_table_utils import *

In [2]:
# --- 日志配置 ---
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

class Config:
    TIMESTAMP = datetime.now().strftime('%y%m%d') 

    IN_DIR = '../data/input'
    IN_OUT_DIR = '../data/in_out'
    OUT_DIR = '../data/output'
    TEMP_DIR = '../tmp_data'



    config_path = '../configs/pretreatment_config.json'

    # 设置一个默认值防止文件读取失败时报错
    _raw_dataset_name = "default_dataset.csv"

    try:
        with open(config_path, 'r', encoding='utf-8') as f:
            _raw_config = json.load(f)
            _raw_dataset_name = _raw_config.get("RAW_DATASET_NAME", _raw_dataset_name)
            
            # 【新增】读取代理并设置环境变量
            _proxy_url = _raw_config.get("PROXY_URL", "").strip()
            if _proxy_url:
                os.environ["http_proxy"] = _proxy_url
                os.environ["https_proxy"] = _proxy_url
                logging.info(f"已成功配置网络代理: {_proxy_url}")
            else:
                logging.info("未配置代理或代理为空，直连网络。")

    except FileNotFoundError:
        logging.warning(f"配置文件未找到: {config_path}")

    # 输入文件路径 (现在使用从 JSON 中读取的文件名)
    RAW_DATASET = os.path.join(IN_DIR, _raw_dataset_name)    
    
    # 【新增】提取不含扩展名的基础文件名 (例如 "data.csv" -> "data")
    _base_name = os.path.splitext(_raw_dataset_name)[0]

    MISSING_DATA_MAG_DIR = os.path.join(TEMP_DIR, _base_name, 'mis_data_mag')
    DEDUPLICATE_DIR = os.path.join(TEMP_DIR, _base_name, 'deduplicate')

    # 输出文件路径 (规则修改：_base_name + _ + TIMESTAMP + ...)
    IDS_MAP_TABLE = os.path.join(IN_OUT_DIR, f'{_base_name}_{TIMESTAMP}_raw-ds_fetch-ids_add-ids.xlsx')
    IDS_MAP_TABLE_OUT = os.path.join(OUT_DIR, f'{_base_name}_{TIMESTAMP}_out_ids-map-tb.xlsx')
    
    BYPRODUCT_TABLE = os.path.join(IN_OUT_DIR, f'{_base_name}_{TIMESTAMP}_ids-map-tb_dd_fg.xlsx')
    BYPRODUCT_TABLE_OUT = os.path.join(OUT_DIR, f'{_base_name}_{TIMESTAMP}_out_byproduct-tb.xlsx')

2026-09-06 17:26:51,230 - INFO - 已成功配置网络代理: http://127.0.0.1:7897


In [3]:
if not os.path.exists(Config.RAW_DATASET):
    logging.error(f"文件未找到: {Config.RAW_DATASET}")
    sys.exit(1)

logging.info("开始加载数据集...")
file_ext = os.path.splitext(Config.RAW_DATASET)[1].lower()
try:
    if file_ext == '.csv':
        raw_dataset = pd.read_csv(Config.RAW_DATASET, encoding='utf-8-sig') 
    elif file_ext in ['.xlsx', '.xls']:
        raw_dataset = pd.read_excel(Config.RAW_DATASET)
except Exception as e:
    print(f"读取文件出错: {e}")
    sys.exit(1)

compound_array = pd.unique(raw_dataset[['c1', 'c2', 'c3']].values.ravel())
compounds_table = pd.DataFrame(compound_array, columns=['compound']).dropna()
compounds_table = compounds_table[compounds_table['compound'].astype(str).str.strip() != '']

total_compounds = len(compounds_table)
logging.info(f"提取到 {total_compounds} 个需要处理的唯一化合物。")

results_dict = {} # 使用字典存储结果，方便后续根据 name 更新

2026-09-06 17:26:51,239 - INFO - 开始加载数据集...
2026-09-06 17:26:51,837 - INFO - 提取到 509 个需要处理的唯一化合物。


In [4]:
# ---------------- Phase 1: 初次全量检索 ----------------
logging.info("=== 启动第一轮全量检索 ===")
for idx, name in enumerate(compounds_table['compound'], start=1):
    print(f"\n" + "="*60)
    print(f"📦 初次检索进度: [{idx}/{total_compounds}] | 当前处理: {name}")
    
    percent = idx / total_compounds
    filled_length = int(40 * percent)
    bar = '█' * filled_length + '-' * (40 - filled_length)
    print(f"🚀 总体进度: |{bar}| {percent*100:.1f}%")
    print(f"--> 正在向 PubChem 请求 ...", flush=True)
    
    res = f_fetch_compound_info(name)
    results_dict[name] = res

2026-09-06 17:26:51,850 - INFO - === 启动第一轮全量检索 ===

📦 初次检索进度: [1/509] | 当前处理: [C2Mim]Cl
🚀 总体进度: |----------------------------------------| 0.2%
--> 正在向 PubChem 请求 ...
2026-09-06 17:26:52,352 - INFO - --> 正在请求: [[C2Mim]Cl] ...
2026-09-06 17:26:53,512 - INFO - PubChem HTTP Error 404 PUGREST.NotFound: No CID found (No CID found that matches the given name)
2026-09-06 17:26:53,513 - INFO - Result '[C2Mim]Cl': ismile=None, uname=None, CAS=None

📦 初次检索进度: [2/509] | 当前处理: ChCl
🚀 总体进度: |----------------------------------------| 0.4%
--> 正在向 PubChem 请求 ...
2026-09-06 17:26:54,014 - INFO - --> 正在请求: [ChCl] ...
2026-09-06 17:26:55,026 - INFO - PubChem HTTP Error 404 PUGREST.NotFound: No CID found (No CID found that matches the given name)
2026-09-06 17:26:55,027 - INFO - Result 'ChCl': ismile=None, uname=None, CAS=None

📦 初次检索进度: [3/509] | 当前处理: [C2OHmim]Cl
🚀 总体进度: |----------------------------------------| 0.6%
--> 正在向 PubChem 请求 ...
2026-09-06 17:26:55,528 - INFO - --> 正在请求: [[C2OHmim]Cl] ...
2

d:\envs\codes\DES-DataAlchemy\dataset_pretreatment\src\make_ids_map_table_utils.py:101: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  data['ismile'] = compound.isomeric_smiles


2026-09-06 17:27:04,006 - INFO -     └─ 成功获取 [ismile]: C(=O)(N)N
2026-09-06 17:27:04,008 - INFO -     └─ 成功获取 [uname]: urea
2026-09-06 17:27:04,008 - INFO -     └─ 成功获取 [cas]: 57-13-6
2026-09-06 17:27:04,009 - INFO - Result 'Urea': ismile=C(=O)(N)N, uname=urea, CAS=57-13-6

📦 初次检索进度: [8/509] | 当前处理: [MeCh]Cl
🚀 总体进度: |----------------------------------------| 1.6%
--> 正在向 PubChem 请求 ...
2026-09-06 17:27:04,510 - INFO - --> 正在请求: [[MeCh]Cl] ...
2026-09-06 17:27:05,441 - INFO - PubChem HTTP Error 404 PUGREST.NotFound: No CID found (No CID found that matches the given name)
2026-09-06 17:27:05,442 - INFO - Result '[MeCh]Cl': ismile=None, uname=None, CAS=None

📦 初次检索进度: [9/509] | 当前处理: [N1111]Cl
🚀 总体进度: |----------------------------------------| 1.8%
--> 正在向 PubChem 请求 ...
2026-09-06 17:27:05,945 - INFO - --> 正在请求: [[N1111]Cl] ...
2026-09-06 17:27:06,886 - INFO - PubChem HTTP Error 404 PUGREST.NotFound: No CID found (No CID found that matches the given name)
2026-09-06 17:27:06,887 - INFO -

In [5]:
# ---------------- Phase 2: 审查与二次回炉 ----------------
failed_names = []
for name, res in results_dict.items():
    # 如果 smiles, uname, cas 中有任何一个是 None，就判定为失败/不完整
    if None in res:
        failed_names.append(name)

if failed_names:
    logging.warning(f"\n⚠️ 第一轮检索结束。发现 {len(failed_names)} 个化合物的数据不完整！准备打回重审...")
    # 给服务器一点喘息的时间，防止是因为被封锁导致的集体失败
    cooldown_time = 10 
    logging.info(f"冷静期：休眠 {cooldown_time} 秒后启动第二轮检索...")
    time.sleep(cooldown_time)
    
    # 清除缓存，强制让这些失败的化合物重新去网络上查一遍，而不是读取刚才失败的记录
    f_fetch_compound_info.cache_clear()
    
    logging.info("=== 启动第二轮补漏检索 (Retry-on-Miss) ===")
    total_retry = len(failed_names)
    for idx, name in enumerate(failed_names, start=1):
        print(f"\n" + "="*60)
        print(f"🚑 补漏进度: [{idx}/{total_retry}] | 正在重新抢救: {name}")
        
        # 在重试前额外增加一个休眠，稳扎稳打
        time.sleep(2) 
        print(f"--> 正在再次向 PubChem 发起请求 ...", flush=True)
        
        retry_res = f_fetch_compound_info(name)
        
        # 覆盖刚才失败的记录
        results_dict[name] = retry_res
        
        if None not in retry_res:
            logging.info(f"🎉 抢救成功！已成功补齐 {name} 的所有字段。")
        else:
            logging.warning(f"❌ 抢救失败，{name} 的数据在 PubChem 中可能确实不存在或存在持续网络阻断。")
else:
    logging.info("\n🎉 第一轮检索非常完美！所有化合物数据均完整，无需重试。")

2026-09-06 17:49:13,268 - WARNING - 
⚠️ 第一轮检索结束。发现 37 个化合物的数据不完整！准备打回重审...
2026-09-06 17:49:13,268 - INFO - 冷静期：休眠 10 秒后启动第二轮检索...
2026-09-06 17:49:23,269 - INFO - === 启动第二轮补漏检索 (Retry-on-Miss) ===

🚑 补漏进度: [1/37] | 正在重新抢救: [C2Mim]Cl
--> 正在再次向 PubChem 发起请求 ...
2026-09-06 17:49:25,774 - INFO - --> 正在请求: [[C2Mim]Cl] ...
2026-09-06 17:49:26,671 - INFO - PubChem HTTP Error 404 PUGREST.NotFound: No CID found (No CID found that matches the given name)
2026-09-06 17:49:26,672 - INFO - Result '[C2Mim]Cl': ismile=None, uname=None, CAS=None
2026-09-06 17:49:26,674 - WARNING - ❌ 抢救失败，[C2Mim]Cl 的数据在 PubChem 中可能确实不存在或存在持续网络阻断。

🚑 补漏进度: [2/37] | 正在重新抢救: ChCl
--> 正在再次向 PubChem 发起请求 ...
2026-09-06 17:49:29,178 - INFO - --> 正在请求: [ChCl] ...
2026-09-06 17:49:30,264 - INFO - PubChem HTTP Error 404 PUGREST.NotFound: No CID found (No CID found that matches the given name)
2026-09-06 17:49:30,266 - INFO - Result 'ChCl': ismile=None, uname=None, CAS=None
2026-09-06 17:49:30,267 - WARNING - ❌ 抢救失败，ChCl 的数据在 P

In [6]:
# ---------------- Phase 3: 整理与保存 ----------------
print(f"\n" + "="*60)
logging.info("所有处理完毕，正在整合并保存数据...")

# 按照原本 DataFrame 的顺序把结果抽出来
final_results = [results_dict[name] for name in compounds_table['compound']]

result_df = pd.DataFrame(
    final_results, 
    index=compounds_table.index, 
    columns=['ismiles', 'uname', 'cas']
)

ids_map_table = pd.concat([compounds_table, result_df], axis=1)


2026-09-06 17:51:36,072 - INFO - 所有处理完毕，正在整合并保存数据...


In [7]:
qlty_sum_output_path = os.path.join(Config.MISSING_DATA_MAG_DIR, 'ids-map-tb_qlty-sum.xlsx')
data_quality_summary(ids_map_table, qlty_sum_output_path)

In [8]:
os.makedirs(os.path.dirname(Config.IDS_MAP_TABLE) or '.', exist_ok=True)
ids_map_table.to_excel(Config.IDS_MAP_TABLE, index=False)

os.makedirs(os.path.dirname(Config.IDS_MAP_TABLE_OUT) or '.', exist_ok=True)
ids_map_table.to_excel(Config.IDS_MAP_TABLE_OUT, index=False)

logging.info(f"✅ 处理完成！结果已成功保存至: {Config.IDS_MAP_TABLE}")

print("请打开in_out文件夹下的raw-ds_fetch-ids_add-ids.xlsx文件，手动填补空缺的标识符")

2026-09-06 17:51:36,344 - INFO - ✅ 处理完成！结果已成功保存至: ../data/in_out\mp_dataset_260906_raw-ds_fetch-ids_add-ids.xlsx
请打开in_out文件夹下的raw-ds_fetch-ids_add-ids.xlsx文件，手动填补空缺的标识符


人工检查数据集质量，填充空值

In [11]:
ids_map_table = pd.read_excel(Config.IDS_MAP_TABLE)
ids_map_table = ids_map_table.dropna()

In [12]:
group_cols = ['ismiles', 'uname', 'cas']
byproduct_table = deduplicate_byproduct_table(ids_map_table, group_cols, Config.DEDUPLICATE_DIR )
byproduct_table.to_excel(Config.BYPRODUCT_TABLE , index=False)
byproduct_table.to_excel(Config.BYPRODUCT_TABLE_OUT , index=False)

涉及重复的总行数: 947
重复明细已整合并保存至: ..\tmp_data\mp_dataset\deduplicate\byproduct_duplicated_details.xlsx (单元格已合并)
开始去重，规则：直接留一
原始行数: 1247
去重后行数: 623
